# Week 6 - Fire localisation model with spatial head

Notebook này huấn luyện lại mô hình nhận diện lửa và hồi quy tọa độ pixel.

Các thay đổi chính:
- Giữ test gốc tách khỏi train/validation để tránh data leakage.
- Horizontal flip cập nhật đồng thời nhãn `x_norm = 1 - x_norm`.
- `BCEWithLogitsLoss` cho confidence và `SmoothL1Loss` cho tọa độ.
- Coordinate loss chỉ tính trên ảnh có lửa.
- Spatial heatmap + soft-argmax thay cho `GAP -> Linear`, giúp giữ thông tin vị trí.
- Log `MAE pixel`, `PCK@10px` và `PCK@25px`.

Checkpoint được lưu vào thư mục `fire-model-data/week6_spatial` và không ghi đè checkpoint cũ.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import random
import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name != 'SAM_Experiment':
    ROOT = Path(r'D:/LAB/SAM_Experiment')
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from train_week6 import (
    seed_everything, load_records, split_records, FireDataset,
    SpatialFireModel, FireLoss, run_epoch, save_checkpoint,
    IMAGE_SIZE, ARCHITECTURE
)

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABELS = ROOT / 'fire-model-data' / 'dataset_labels (1).json'
DATASET_ROOT = ROOT / 'fire-detection-from-cctv'
OUTPUT_DIR = ROOT / 'fire-model-data' / 'week6_spatial'
EPOCHS = 30
BATCH_SIZE = 32
NUM_WORKERS = 2 if DEVICE.type == 'cuda' else 0
LEARNING_RATE = 1e-4
TRAIN_ARGS = SimpleNamespace(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE)
USE_PRETRAINED = True
print('device =', DEVICE)
print('architecture =', ARCHITECTURE)
print('labels =', LABELS)
print('output =', OUTPUT_DIR)

## 1. Load dữ liệu và kiểm tra split

Nếu metadata có thư mục `train/test`, test gốc được giữ nguyên. Chỉ dữ liệu train gốc được tách tiếp thành train/validation. Các bản ghi trùng vật lý được loại trước khi chia.

In [ ]:
records, load_stats = load_records(LABELS, DATASET_ROOT)
splits = split_records(records, seed=42)
split_info = {
    name: {
        'count': len(items),
        'fire': sum(r.has_fire for r in items),
        'no_fire': sum(not r.has_fire for r in items),
    }
    for name, items in splits.items()
}
print('load_stats =', load_stats)
print('split_info =', split_info)
assert all(split_info[k]['count'] > 0 for k in ('train', 'val', 'test'))

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    FireDataset(splits['train'], train=True),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    FireDataset(splits['val'], train=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
test_loader = DataLoader(
    FireDataset(splits['test'], train=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
images, targets, sizes, paths = next(iter(train_loader))
print('image batch:', tuple(images.shape))
print('target batch:', tuple(targets.shape))
print('first target [has_fire, x_norm, y_norm]:', targets[0].tolist())

## 2. Smoke test spatial head, loss và backward

Cell này cần chạy thành công trước khi bắt đầu train dài.

In [ ]:
smoke_model = SpatialFireModel(pretrained=False).to(DEVICE)
smoke_loss = FireLoss(lambda_coord=5.0, lambda_heatmap=0.5)
smoke_images = images[:2].to(DEVICE)
smoke_targets = targets[:2].to(DEVICE)
smoke_outputs = smoke_model(smoke_images)
smoke_losses = smoke_loss(smoke_outputs, smoke_targets)
smoke_losses['total'].backward()
print({k: float(v.detach().cpu()) for k, v in smoke_losses.items()})
print({k: tuple(v.shape) for k, v in smoke_outputs.items()})
print('smoke test: OK')

## 3. Khởi tạo mô hình và optimizer

Dùng pretrained MobileNetV4 khi train thật. `pretrained=False` chỉ nên dùng để smoke test.

In [ ]:
model = SpatialFireModel(pretrained=USE_PRETRAINED).to(DEVICE)
criterion = FireLoss(lambda_coord=5.0, lambda_heatmap=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, max(1, EPOCHS), eta_min=LEARNING_RATE / 100
)
print('trainable parameters =', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Training loop

Model tốt nhất được chọn theo validation total loss. Các metric tọa độ được log riêng để không nhầm classification accuracy với localization quality.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_val = float('inf')
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_metric = run_epoch(
        model, train_loader, criterion, DEVICE, optimizer=optimizer
    )
    val_loss, val_metric = run_epoch(
        model, val_loader, criterion, DEVICE
    )
    scheduler.step()

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'train_metric': train_metric,
        'val_metric': val_metric,
    }
    history.append(row)
    print(
        f"epoch={epoch:03d} train={train_loss['total']:.5f} "
        f"val={val_loss['total']:.5f} "
        f"MAE={val_metric['mae_px']:.2f}px "
        f"PCK10={val_metric['pck10']:.3f} "
        f"PCK25={val_metric['pck25']:.3f}"
    )

    if val_loss['total'] < best_val:
        best_val = val_loss['total']
        save_checkpoint(
            OUTPUT_DIR / 'best_spatial.pth', model, optimizer, scheduler,
            epoch, best_val, split_info,
            TRAIN_ARGS
        )

    save_checkpoint(
        OUTPUT_DIR / 'last_spatial.pth', model, optimizer, scheduler,
        epoch, val_loss['total'], split_info,
        TRAIN_ARGS
    )
    (OUTPUT_DIR / 'history_notebook.json').write_text(
        json.dumps(history, indent=2), encoding='utf-8'
    )

print('best validation loss =', best_val)
print('best checkpoint =', OUTPUT_DIR / 'best_spatial.pth')

## 5. Đánh giá trên test độc lập

Không dùng test loss để chọn checkpoint. Cell này chỉ chạy sau khi train xong và dùng test gốc độc lập.

In [ ]:
checkpoint = torch.load(OUTPUT_DIR / 'best_spatial.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model'])
model.eval()
test_loss, test_metric = run_epoch(model, test_loader, criterion, DEVICE)
print('test_loss =', test_loss)
print('test_metric =', test_metric)
(OUTPUT_DIR / 'test_metrics_notebook.json').write_text(
    json.dumps({'loss': test_loss, 'metric': test_metric}, indent=2),
    encoding='utf-8'
)

## 6. Vẽ learning curve

Nếu validation loss giảm nhưng MAE/PCK không cải thiện, cần ưu tiên điều chỉnh nhãn hoặc loss tọa độ thay vì chỉ tăng số epoch.

In [ ]:
epochs = [r['epoch'] for r in history]
train_total = [r['train_loss']['total'] for r in history]
val_total = [r['val_loss']['total'] for r in history]
val_mae = [r['val_metric']['mae_px'] for r in history]
val_pck10 = [r['val_metric']['pck10'] for r in history]
val_pck25 = [r['val_metric']['pck25'] for r in history]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs, train_total, label='train')
axes[0].plot(epochs, val_total, label='val')
axes[0].set_title('Total loss')
axes[0].legend()
axes[1].plot(epochs, val_mae, color='tab:orange')
axes[1].set_title('Validation MAE (px)')
axes[2].plot(epochs, val_pck10, label='PCK@10')
axes[2].plot(epochs, val_pck25, label='PCK@25')
axes[2].set_ylim(0, 1)
axes[2].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curve_spatial.png', dpi=150)
plt.show()

## Lưu ý khi tích hợp vào pipeline 3D

`best_spatial.pth` không tương thích trực tiếp với `fire_detector.py` cũ vì head đã thay đổi. Cần viết adapter inference tương ứng với `SpatialFireModel`, sau đó mới nối `coord` vào `CameraGeometry.pixel_to_ray()`.

Không dùng `ground_truth.json` của 10 ảnh thủ công để chọn checkpoint trong notebook này; file đó dành cho đánh giá localization riêng sau khi mô hình 2D đã được train.